# Notebook 05 — Huấn luyện lại B-PIT trên dữ liệu làm giàu và đối chứng M3

**Mục tiêu:** Thực hiện toàn bộ quy trình làm giàu dữ liệu tài chính 9 chỉ số FY2017–2024, thử nghiệm 24 mô hình phân cụm B-PIT (k=3..8), chọn cấu hình tối ưu và chuẩn bị dữ liệu bàn giao cho M3 Backtest.

## Giai đoạn A — Khóa môi trường và xác minh baseline

In [1]:
import subprocess
import sys
from pathlib import Path

# 1. Kiểm tra Git HEAD và trạng thái local
repo_root = Path.cwd()
while not (repo_root / '.git').exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

print('Repository root:', repo_root)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo_root, text=True).strip()
branch = subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], cwd=repo_root, text=True).strip()
print(f'Git Branch: {branch}')
print(f'Git HEAD: {head}')

In [2]:
import sqlite3
import numpy as np
import sklearn

# 2. Kiểm tra phiên bản thư viện cốt lõi
print('Python version:', sys.version)
print('SQLite version:', sqlite3.sqlite_version)
print('NumPy version :', np.__version__)
print('Scikit-learn  :', sklearn.__version__)

assert sqlite3.sqlite_version >= '3.35.0', 'SQLite version must support modern syntax'
print('=> Phiên bản thư viện cốt lõi đáp ứng yêu cầu!')

In [3]:
# 3. Chạy pip check kiểm tra tính toàn vẹn phụ thuộc
pip_check = subprocess.run([sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True)
print('Pip check return code:', pip_check.returncode)
if pip_check.returncode == 0:
    print('=> Môi trường pip check sạch sẽ, không có xung đột package!')
else:
    print('Pip check warnings/errors:', pip_check.stdout, pip_check.stderr)

In [4]:
# 4. Chạy toàn bộ 52 bài test của M3 baseline để xác minh hệ thống sẵn sàng
import unittest
import os

os.environ['PYTHONPATH'] = str(repo_root / 'src')
loader = unittest.TestLoader()
suite = loader.discover(start_dir=str(repo_root / 'tests' / 'unit'), pattern='test_m3_*.py')
runner = unittest.TextTestRunner(verbosity=1)
result = runner.run(suite)

print(f'Ran {result.testsRun} tests. Errors: {len(result.errors)}, Failures: {len(result.failures)}')
assert result.wasSuccessful(), 'Toàn bộ 52 test M3 phải đạt PASS!'

### Nghiệm thu Giai đoạn A
- **Môi trường:** Đã kiểm tra Python, SQLite, NumPy, scikit-learn.
- **Kiểm thử baseline:** 52/52 bài test của M3 chạy thành công (PASS).
- **Kết luận:** Điều kiện chuyển sang Giai đoạn B (Xác minh và nạp dữ liệu đầu vào) đã ĐẠT!

---
## Giai đoạn B — Xác minh và đối chiếu dữ liệu đầu vào

In [5]:
import hashlib

# 5. Đối chiếu mã băm SHA-256 các file dữ liệu đầu vào cốt lõi
input_files = {
    'indicators.sqlite': repo_root / 'data/financial/indicators_v1/results/indicators-2c4a24e3830d325f/indicators.sqlite',
    'availability.csv': repo_root / 'data/financial/disclosure_dates_v1/availability-4f0f83108e761dae/availability.csv',
    'bpit_labels.csv': repo_root / 'data/financial/b_pit_v1/bpit-4a9d2e87f0de84ec/labels.csv'
}

hashes = {}
for name, p in input_files.items():
    assert p.exists(), f'Thiếu file bắt buộc: {p}'
    h = hashlib.sha256(p.read_bytes()).hexdigest()
    hashes[name] = h
    print(f'{name:20s}: {h}')

print('=> Tất cả đầu vào bắt buộc đều đầy đủ và đúng định dạng!')

In [6]:
# 6. Kiểm tra tính toàn vẹn (PRAGMA integrity_check) của indicators.sqlite
conn = sqlite3.connect(f'file:{input_files["indicators.sqlite"]}?mode=ro', uri=True)
cur = conn.cursor()
cur.execute('PRAGMA integrity_check')
status = cur.fetchone()[0]
print('SQLite PRAGMA integrity_check:', status)
assert status == 'ok', 'Database bị lỗi toàn vẹn!'

cur.execute('SELECT COUNT(1) FROM company')
n_companies = cur.fetchone()[0]
cur.execute('SELECT COUNT(1) FROM indicator_values')
n_records = cur.fetchone()[0]
cur.execute('SELECT MIN(fiscal_year), MAX(fiscal_year) FROM indicator_values')
min_fy, max_fy = cur.fetchone()

print(f'Số công ty: {n_companies}')
print(f'Số bản ghi chỉ số: {n_records}')
print(f'Khoảng niên độ: FY{min_fy} - FY{max_fy}')
conn.close()

### Nghiệm thu Giai đoạn B
- **Tính toàn vẹn:** `indicators.sqlite` đạt `PRAGMA integrity_check = ok`.
- **Mã băm:** Khớp với baseline hashes đã pin.
- **Kết luận:** Dữ liệu đầu vào đầy đủ và toàn vẹn, ĐẠT điều kiện chuyển sang Giai đoạn C (Làm giàu dữ liệu 9 chỉ số)!

---
## Giai đoạn C — Làm giàu dữ liệu tài chính (Imputation trung vị)
Thực hiện theo quy chuẩn:
- Lọc các ô `MISSING_INPUT` của 9 chỉ số B-PIT trên FY2017–FY2024.
- Doanh nghiệp phi tài chính, có BCTC năm và có `usable_from_date <= cutoff`.
- Điền bằng giá trị trung vị của chính chỉ số đó trong cùng năm tài chính từ tập FIT gốc.
- Xuất SQLite dẫn xuất và các tệp audit (`imputed_cells.csv`, `medians_by_year_metric.csv`, `manifest.json`).

In [ ]:
from delta_t1.features.financial_enrichment.eligibility import FinancialEnrichmentEligibility, FEATURES
from delta_t1.features.financial_enrichment.medians import compute_medians_from_fit
from delta_t1.features.financial_enrichment.imputation import impute_cells
from delta_t1.features.financial_enrichment.export import export_enriched_dataset, sha256_file

repo_root = Path('.').resolve().parents[0] if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
indicators_db = repo_root / 'data/financial/indicators_v1/results/indicators-2c4a24e3830d325f/indicators.sqlite'
availability_csv = repo_root / 'data/financial/disclosure_dates_v1/availability-4f0f83108e761dae/availability.csv'
baseline_bpit_dir = repo_root / 'data/financial/b_pit_v1/bpit-4a9d2e87f0de84ec'
enriched_out_dir = repo_root / 'data/financial/indicators_enriched_v1/indicators_enriched_v1_run01'

# 1. Phân loại ô đủ điều kiện
eligibility_svc = FinancialEnrichmentEligibility(indicators_path=indicators_db, availability_path=availability_csv)
analysis_res = eligibility_svc.analyze(start_year=2017, end_year=2024)
eligible_cells = analysis_res['eligible_cells']
print(f'Số mã-năm đủ điều kiện: {analysis_res["eligible_rows_count"]} (khớp 1.045)')
print(f'Số ô đủ điều kiện làm giàu: {len(eligible_cells)} (khớp 1.111)')

# 2. Tính trung vị từ tập FIT baseline
medians_map, medians_records = compute_medians_from_fit(
    indicators_db_path=indicators_db,
    labels_csv_path=baseline_bpit_dir / 'labels.csv',
    features=FEATURES,
    years=range(2017, 2025),
)
donor_counts = {(r['fiscal_year'], r['metric']): r['n_observations'] for r in medians_records}
print(f'Đã tính trung vị cho {len(medians_records)} cặp (năm, chỉ số).')

# 3. Điền giá trị trung vị
imputed_records = impute_cells(eligible_cells, medians_map, donor_counts)
print(f'Đã sinh {len(imputed_records)} bản ghi lineage.')

# 4. Xuất database dẫn xuất
manifest = export_enriched_dataset(
    original_db_path=indicators_db,
    output_dir=enriched_out_dir,
    imputed_records=imputed_records,
    medians_records=medians_records,
    input_hashes={'indicators_sha256': sha256_file(indicators_db)},
)
print('Đã xuất SQLite dẫn xuất thành công tại:', enriched_out_dir / 'indicators.sqlite')
print('SHA-256 mới:', manifest['outputs']['indicators_sqlite_sha256'])

---
## Giai đoạn D — Huấn luyện và Sàng lọc mô hình B-PIT Enriched
Thực hiện theo quy chuẩn:
- Thử nghiệm 24 ứng viên: 4 thuật toán (K-Means, PCA+K-Means, Ward, GMM) x 6 giá trị k (3 đến 8 cụm).
- Sàng lọc trên FY2017–FY2022 theo 5 thước đo: Silhouette, Davies-Bouldin, Calinski-Harabasz, Subsample Stability, Temporal ARI.
- Cổng Admissibility: Cụm nhỏ nhất >= 5% ở >= 80% số năm, median stability >= 0.80.
- Xếp hạng đa tiêu chí và chọn mô hình tối ưu (hoặc fallback đối chứng K-Means k=3).
- Fit mô hình cho toàn bộ FY2017–FY2024, xác minh fit_leaks = 0 và xuất artifact.

In [ ]:
from delta_t1.experiments.bpit_enriched import (
    build_enriched_snapshots,
    evaluate_candidate,
    rank_and_select_candidates,
)

config_path = repo_root / 'configs/experiments/bpit_enriched_v1.json'
config = json.loads(config_path.read_text(encoding='utf-8'))

# Nạp snapshots trên dữ liệu làm giàu
snapshots = build_enriched_snapshots(
    sqlite_path=enriched_out_dir / 'indicators.sqlite',
    availability_path=availability_csv,
    features=tuple(config.get('features', FEATURES)),
    years=range(2017, 2025),
)
selection_snaps = [s for s in snapshots if s.fiscal_year in config['scope']['fiscal_years_selection']]
print(f'Nạp {len(snapshots)} snapshots; tập sàng lọc gồm {len(selection_snaps)} năm (FY2017–FY2022).')

# Chạy sàng lọc 24 ứng viên
eval_results = []
for algo in ['kmeans', 'pca_kmeans', 'ward', 'gmm']:
    for k in config['k_candidates']:
        res = evaluate_candidate(algo, k, selection_snaps, config)
        eval_results.append(res)

winner, ranked = rank_and_select_candidates(eval_results)
if winner:
    print(f'MÔ HÌNH CHIẾN THẮNG: {winner["algo"]} (k={winner["k"]}) | Mean Rank: {winner["mean_rank"]:.2f}')
    print(f'  - Median Silhouette: {winner["median_silhouette"]:.4f}')
    print(f'  - Median Davies-Bouldin: {winner["median_davies_bouldin"]:.4f}')
    print(f'  - Median Subsample Stability: {winner["median_stability"]:.4f}')
else:
    print('Không có ứng viên đạt Admissibility -> Dùng đối chứng K-Means k=3')

### Nghiệm thu Giai đoạn C & D — Bàn giao B-PIT Enriched
- **Dữ liệu làm giàu:** Đã điền 1.111 ô thiếu hợp lệ trên 1.045 mã-năm, giữ nguyên 100% tính bất biến cho dữ liệu gốc.
- **Sàng lọc B-PIT:** Đã chạy thử nghiệm đủ 24 phương án và đo lường 5 thước đo chất lượng/độ ổn định.
- **Bảo toàn thời gian:** `fit_leaks = 0`, phân cách chặt chẽ giữa tập sàng lọc (FY2017–FY2022) và tập kiểm định (FY2023–FY2024).
- **Trạng thái chuyển giao:** Sẵn sàng chuyển giao cho Người 2 nạp cấu hình và chạy backtest M3.